In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

train = pd.read_csv('train_ctrUa4K.csv')
test = pd.read_csv('test_lAUu6dG.csv')

print("Training Data Rows and Columns:", train.shape)
train.head()

Training Data Rows and Columns: (614, 13)


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [11]:
# See which columns have empty (NaN) cells
print(train.isnull().sum())

# See how many loans were approved vs rejected
print(train['Loan_Status'].value_counts(normalize=True))

Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64
Loan_Status
Y    0.687296
N    0.312704
Name: proportion, dtype: float64


In [12]:
# Fix Categorical Missing Values
train['Gender'].fillna(train['Gender'].mode()[0], inplace=True)
train['Married'].fillna(train['Married'].mode()[0], inplace=True)
train['Dependents'].fillna(train['Dependents'].mode()[0], inplace=True)
train['Self_Employed'].fillna(train['Self_Employed'].mode()[0], inplace=True)
train['Credit_History'].fillna(train['Credit_History'].mode()[0], inplace=True)
train['Loan_Amount_Term'].fillna(train['Loan_Amount_Term'].mode()[0], inplace=True)

# Fix Numerical Missing Value
train['LoanAmount'].fillna(train['LoanAmount'].median(), inplace=True)

# --- DO THE SAME FOR THE TEST DATA ---
test['Gender'].fillna(test['Gender'].mode()[0], inplace=True)
test['Married'].fillna(test['Married'].mode()[0], inplace=True)
test['Dependents'].fillna(test['Dependents'].mode()[0], inplace=True)
test['Self_Employed'].fillna(test['Self_Employed'].mode()[0], inplace=True)
test['Credit_History'].fillna(test['Credit_History'].mode()[0], inplace=True)
test['Loan_Amount_Term'].fillna(test['Loan_Amount_Term'].mode()[0], inplace=True)
test['LoanAmount'].fillna(test['LoanAmount'].median(), inplace=True)

In [13]:
# Drop Loan_ID because it's just a name/ID and doesn't help predict anything
train = train.drop('Loan_ID', axis=1)
test_ids = test['Loan_ID'] # Save this for later
test = test.drop('Loan_ID', axis=1)

# Separate the Target (Loan_Status) from the Features
X = train.drop('Loan_Status', axis=1)
y = train['Loan_Status']

# Convert all text columns into numbers automatically
X = pd.get_dummies(X)
test = pd.get_dummies(test)

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Split your training data into two: one to train, one to check accuracy
x_train, x_cv, y_train, y_cv = train_test_split(X, y, test_size=0.3, random_state=1)

# Initialize and Train
model = LogisticRegression()
model.fit(x_train, y_train)

# Check Accuracy on the validation set
pred_cv = model.predict(x_cv)
print("Accuracy Score:", accuracy_score(y_cv, pred_cv))

Accuracy Score: 0.7945945945945946


In [15]:
# Predict on the test data
pred_test = model.predict(test)

# Create the submission dataframe
submission = pd.DataFrame({
    'Loan_ID': test_ids,
    'Loan_Status': pred_test
})

# Save to CSV
submission.to_csv('logistic_submission.csv', index=False)
print("Submission file 'logistic_submission.csv' is ready!")

Submission file 'logistic_submission.csv' is ready!
